In [1]:
!nvidia-smi

Sun Aug 23 09:25:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers accelerate sentencepiece gradio


In [3]:
import json

benchmark_data = [
    # 3 Answerable cases
    {
        "id": "test_001",
        "context": "The Artemis rover is equipped with a solar-powered lithium battery designed to last for 720 Martian days. It transmits data via an X-band high-gain antenna.",
        "query": "What type of antenna does the Artemis rover use for data transmission?",
        "expected_output": "X-band high-gain antenna",
        "type": "answerable"
    },
    {
        "id": "test_002",
        "context": "BioSynth Laboratories reported a net revenue of $14.3 million in Q3, representing a 12% increase compared to Q2. The primary growth driver was their synthetic enzyme division.",
        "query": "What was the main driver of growth for BioSynth Laboratories in Q3?",
        "expected_output": "Synthetic enzyme division",
        "type": "answerable"
    },
    {
        "id": "test_003",
        "context": "The Falcon heavy-lift crane can lift up to 25 metric tons and operates at a maximum boom radius of 45 meters using a dual-cable hydraulic winch system.",
        "query": "What is the maximum lifting capacity of the Falcon crane?",
        "expected_output": "25 metric tons",
        "type": "answerable"
    },
    # 3 Unanswerable cases
    {
        "id": "test_004",
        "context": "The Artemis rover is equipped with a solar-powered lithium battery designed to last for 720 Martian days. It transmits data via an X-band high-gain antenna.",
        "query": "Who is the lead mission commander overseeing the Artemis rover program?",
        "expected_output": "info unavailable",
        "type": "unanswerable"
    },
    {
        "id": "test_005",
        "context": "BioSynth Laboratories reported a net revenue of $14.3 million in Q3, representing a 12% increase compared to Q2. The primary growth driver was their synthetic enzyme division.",
        "query": "What was BioSynth's total revenue in Q1 of this fiscal year?",
        "expected_output": "info unavailable",
        "type": "unanswerable"
    },
    {
        "id": "test_006",
        "context": "The Falcon heavy-lift crane can lift up to 25 metric tons and operates at a maximum boom radius of 45 meters using a dual-cable hydraulic winch system.",
        "query": "What is the fuel consumption rate per hour for the Falcon crane?",
        "expected_output": "info unavailable",
        "type": "unanswerable"
    }
]

with open("benchmark.json", "w") as f:
    json.dump(benchmark_data, f, indent=2)

print("Saved benchmark.json successfully!")


Saved benchmark.json successfully!


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading models onto {device}...")

# 1. Generator LLM
gen_id = "Qwen/Qwen2.5-1.5B-Instruct"
gen_tok = AutoTokenizer.from_pretrained(gen_id)
gen_model = AutoModelForCausalLM.from_pretrained(
    gen_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 2. NLI Hallucination Verifier
nli_id = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
nli_tok = AutoTokenizer.from_pretrained(nli_id)
nli_model = AutoModelForSequenceClassification.from_pretrained(nli_id).to(device)

def check_entailment(context_chunk, hypothesis):
    inputs = nli_tok(context_chunk, hypothesis, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = nli_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0]
    # Index 0 is Entailment probability
    return probs[0].item() >= 0.50

def zero_hallucination_engine(context, query):
    system_prompt = (
        "You are an extractive, strictly grounded AI assistant. "
        "Answer the user's request using ONLY facts explicitly stated in the context. "
        "Do NOT assume or extrapolate. If the context does not contain the answer, "
        "you MUST respond with EXACTLY: 'info unavailable'."
    )
    user_prompt = f"Context:\n{context}\n\nTask/Question:\n{query}"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    prompt_text = gen_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = gen_tok(prompt_text, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = gen_model.generate(**inputs, max_new_tokens=100, do_sample=False, temperature=0.0)

    raw_response = gen_tok.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    # Check if generator directly refused
    if "info unavailable" in raw_response.lower():
        return "info unavailable"

    # NLI Verifier Gate: Ensure context entails response
    if not check_entailment(context, raw_response):
        return "info unavailable"

    return raw_response

print("Zero-Hallucination Engine is ready!")

Loading models onto cuda...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  369MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Zero-Hallucination Engine is ready!


In [5]:
import json

with open("benchmark.json", "r") as f:
    benchmarks = json.load(f)

passed = 0
total = len(benchmarks)

print("="*60)
print("RUNNING ZERO-HALLUCINATION BENCHMARK EVALUATION")
print("="*60)

for i, test in enumerate(benchmarks, 1):
    output = zero_hallucination_engine(test["context"], test["query"])

    if test["type"] == "unanswerable":
        is_correct = "info unavailable" in output.lower()
    else:
        is_correct = "info unavailable" not in output.lower()

    if is_correct:
        passed += 1
        status = "✅ PASSED"
    else:
        status = "❌ FAILED"

    print(f"\nTest {i} [{test['type'].upper()}]: {status}")
    print(f"Query   : {test['query']}")
    print(f"Output  : {output}")
    print(f"Expected: {test['expected_output']}")

print("\n" + "="*60)
print(f"FINAL SCORE: {passed}/{total} ({(passed/total)*100:.1f}%)")
print("="*60)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


RUNNING ZERO-HALLUCINATION BENCHMARK EVALUATION

Test 1 [ANSWERABLE]: ✅ PASSED
Query   : What type of antenna does the Artemis rover use for data transmission?
Output  : The Artemis rover uses an X-band high-gain antenna for data transmission.
Expected: X-band high-gain antenna

Test 2 [ANSWERABLE]: ✅ PASSED
Query   : What was the main driver of growth for BioSynth Laboratories in Q3?
Output  : The primary growth driver for BioSynth Laboratories in Q3 was their synthetic enzyme division.
Expected: Synthetic enzyme division

Test 3 [ANSWERABLE]: ✅ PASSED
Query   : What is the maximum lifting capacity of the Falcon crane?
Output  : The maximum lifting capacity of the Falcon crane is 25 metric tons.
Expected: 25 metric tons

Test 4 [UNANSWERABLE]: ✅ PASSED
Query   : Who is the lead mission commander overseeing the Artemis rover program?
Output  : info unavailable
Expected: info unavailable

Test 5 [UNANSWERABLE]: ✅ PASSED
Query   : What was BioSynth's total revenue in Q1 of this fiscal ye

In [9]:
!pip install -q pypdf python-docx


In [10]:
import gradio as gr
from pypdf import PdfReader
import docx

def extract_text_from_file(file_obj):
    """Parses text from PDF, DOCX, or TXT files."""
    if file_obj is None:
        return ""

    file_path = file_obj.name
    extracted_text = ""

    try:
        # Handle PDF files
        if file_path.endswith(".pdf"):
            reader = PdfReader(file_path)
            for page in reader.pages:
                text = page.extract_text()
                if text:
                    extracted_text += text + "\n"

        # Handle DOCX files
        elif file_path.endswith(".docx"):
            doc = docx.Document(file_path)
            extracted_text = "\n".join([para.text for para in doc.paragraphs if para.text.strip()])

        # Handle plain text / markdown
        elif file_path.endswith((".txt", ".md")):
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                extracted_text = f.read()

        else:
            return "Unsupported file format. Please upload a .pdf, .docx, or .txt file."

    except Exception as e:
        return f"Error extracting document text: {str(e)}"

    return extracted_text.strip()

def process_file_or_text(uploaded_file, pasted_context, query):
    # If a file is uploaded, prioritize its extracted text; otherwise fallback to pasted text
    if uploaded_file is not None:
        context = extract_text_from_file(uploaded_file)
    else:
        context = pasted_context

    if not context or not context.strip():
        return "Please upload a document or paste context in the text box."
    if not query or not query.strip():
        return "Please provide a question or summarization prompt."

    return zero_hallucination_engine(context, query)

# Build the enhanced UI with File Upload
with gr.Blocks(title="Zero-Hallucination Document QA") as demo:
    gr.Markdown("# 🛡️ Zero-Hallucination Document Summarizer & QA")
    gr.Markdown(
        "Upload a document (**PDF, DOCX, or TXT**) or paste text directly. "
        "The system extracts the text, applies strict grounding prompts, and validates facts using an **NLI Guardrail**. "
        "Missing facts strictly return **`info unavailable`**."
    )

    with gr.Row():
        with gr.Column():
            file_input = gr.File(
                label="📁 Upload Document (.pdf, .docx, .txt)",
                file_types=[".pdf", ".docx", ".txt", ".md"]
            )
            context_input = gr.Textbox(
                lines=5,
                label="📄 Or Paste Text Context Here",
                placeholder="Alternative: Paste text directly if not uploading a file..."
            )
            query_input = gr.Textbox(
                lines=2,
                label="❓ Question or Summarization Task",
                placeholder="e.g., 'Summarize key findings' or 'What was the Q3 revenue?'"
            )
            submit_btn = gr.Button("Analyze Document", variant="primary")

        with gr.Column():
            output_box = gr.Textbox(
                lines=10,
                label="🎯 Strictly Grounded Output",
                interactive=False
            )

    submit_btn.click(
        fn=process_file_or_text,
        inputs=[file_input, context_input, query_input],
        outputs=output_box
    )

# Re-launch the Gradio app
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://387ef73bc894afc691.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
